# KRNN WRDS connection diagnostics

Use this notebook to isolate where WRDS access is failing on this machine.

Order of operations:
1. Load project config and credential env settings.
2. Verify raw host/port reachability.
3. Test a direct `psycopg2` connection to WRDS.
4. Optionally create/update `pgpass.conf` after a successful direct login.
5. Test the project's `WRDSClient` wrapper using the same credentials.
6. Inspect visible CRSP tables and run a small pilot pull.

If step 3 fails, the issue is not the KRNN code. It is WRDS auth/session/account/network.
If step 3 succeeds but step 5 fails, the issue is in the Python client layer and we should bypass the `wrds` package in project code.

In [1]:
import os
import sys
import socket
import getpass
from pathlib import Path

import pandas as pd
import psycopg2
import yaml

WRDS_USERNAME_OVERRIDE = None
WRDS_PASSWORD_OVERRIDE = None  # Leave None to get a password prompt
WRDS_HOST = 'wrds-pgdata.wharton.upenn.edu'
WRDS_PORT = 9737
WRDS_DBNAME = 'wrds'

cfg_path = Path('config_v5.yaml')
if not cfg_path.exists():
    raise FileNotFoundError(
        f'config_v5.yaml not found in CWD={Path.cwd()}. Open the repo root before running the notebook.'
    )

ROOT = cfg_path.resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
username_env = cfg.get('wrds', {}).get('username_env', 'WRDS_USERNAME')
password_env = cfg.get('wrds', {}).get('password_env', 'WRDS_PASSWORD')

if WRDS_USERNAME_OVERRIDE:
    os.environ[username_env] = WRDS_USERNAME_OVERRIDE
if WRDS_PASSWORD_OVERRIDE:
    os.environ[password_env] = WRDS_PASSWORD_OVERRIDE

username = cfg.get('wrds', {}).get('username') or os.getenv(username_env)
password = cfg.get('wrds', {}).get('password') or os.getenv(password_env)

print('ROOT:', ROOT)
print('Configured data source:', cfg.get('data', {}).get('source'))
print('WRDS host:', WRDS_HOST)
print('WRDS port:', WRDS_PORT)
print('Username present:', bool(username))
print('Password present in env/config:', bool(password))

ROOT: D:\PyCharmProjects\KRNN
Configured data source: wrds
WRDS host: wrds-pgdata.wharton.upenn.edu
WRDS port: 9737
Username present: True
Password present in env/config: False


In [2]:
sock = socket.create_connection((WRDS_HOST, WRDS_PORT), timeout=10)
sock.close()
print(f'TCP reachability OK: {WRDS_HOST}:{WRDS_PORT}')

TCP reachability OK: wrds-pgdata.wharton.upenn.edu:9737


## Fresh MFA reminder

Before running the next cell, do all of the following:

- stay on the same network/IP
- keep VPN off unless WRDS requires it for your institution
- if needed, do a fresh Duo-authenticated WRDS web or SSH login first
- have your WRDS password ready

The next cell tests PostgreSQL directly, without the `wrds` package.

In [3]:
if not username:
    username = input(f'Enter WRDS username [{getpass.getuser()}]: ').strip() or getpass.getuser()
if not password:
    password = getpass.getpass('Enter WRDS password (not stored): ')

print('Using username:', username)
print('Password captured:', bool(password))

Using username: ajcolmenar
Password captured: True


In [4]:
direct_conn = None
direct_result = None

try:
    direct_conn = psycopg2.connect(
        host=WRDS_HOST,
        port=WRDS_PORT,
        dbname=WRDS_DBNAME,
        user=username,
        password=password,
        sslmode='require',
        connect_timeout=15,
        application_name='KRNN psycopg2 diagnostic',
    )
    with direct_conn.cursor() as cur:
        cur.execute('select current_user, current_database(), version()')
        direct_result = cur.fetchone()
    print('Direct psycopg2 connection succeeded.')
    print('current_user:', direct_result[0])
    print('current_database:', direct_result[1])
    print('server_version:', direct_result[2][:120], '...')
except Exception as exc:
    print('Direct psycopg2 connection failed.')
    print(repr(exc))
    raise
finally:
    if direct_conn is not None:
        direct_conn.close()

Direct psycopg2 connection succeeded.
current_user: ajcolmenar
current_database: wrds
server_version: PostgreSQL 17.7 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 11.5.0 20240719 (Red Hat 11.5.0-11), 64-bit ...


In [5]:
WRITE_PGPASS = False

def escape_pgpass(value: str) -> str:
    return value.replace('\\', '\\\\').replace(':', '\\:')

if WRITE_PGPASS:
    appdata = os.getenv('APPDATA')
    if not appdata:
        raise RuntimeError('APPDATA is not set; cannot write pgpass.conf on Windows.')
    pgdir = Path(appdata) / 'postgresql'
    pgdir.mkdir(parents=True, exist_ok=True)
    pgpass_path = pgdir / 'pgpass.conf'
    new_line = f'{WRDS_HOST}:{WRDS_PORT}:{WRDS_DBNAME}:{username}:{escape_pgpass(password)}\n'
    existing = []
    if pgpass_path.exists():
        existing = pgpass_path.read_text(encoding='utf-8').splitlines(keepends=True)
    filtered = []
    for line in existing:
        if not line.startswith(f'{WRDS_HOST}:{WRDS_PORT}:{WRDS_DBNAME}:{username}:'):
            filtered.append(line)
    filtered.append(new_line)
    pgpass_path.write_text(''.join(filtered), encoding='utf-8')
    print('Wrote', pgpass_path)
else:
    print('Set WRITE_PGPASS = True in this cell if you want to create/update pgpass.conf.')


Set WRITE_PGPASS = True in this cell if you want to create/update pgpass.conf.


In [6]:
from src.data.wrds_client import WRDSClient

client = None

try:
    client = WRDSClient(
        wrds_username=username,
        wrds_password=password,
        autoconnect=True,
        verbose=bool(cfg.get('wrds', {}).get('verbose', False)),
    )
    libraries = client.connect().list_libraries()
    print('WRDSClient connection succeeded.')
    print('Visible libraries (first 20):', libraries[:20])
except Exception as exc:
    print('WRDSClient/wrds package connection failed even though direct psycopg2 passed.')
    print(repr(exc))
    raise

Loading library list...
Done
WRDSClient connection succeeded.
Visible libraries (first 20): ['aha_sample', 'ahasamp', 'audit', 'audit_audit_comp', 'audit_common', 'audit_corp_legal', 'auditsmp', 'auditsmp_all', 'bank', 'bank_all', 'bank_premium_samp', 'banksamp', 'block', 'block_all', 'boardex', 'boardex_na', 'boardex_trial', 'boardsmp', 'bvd_amadeus_trial', 'bvd_bvdbankf_trial']


In [7]:
from src.data.wrds_equity_source import WRDSEquitySource

schema = cfg.get('wrds', {}).get('schema', 'crsp')
candidate_groups = {
    'names': cfg.get('wrds', {}).get('names_table_candidates', []),
    'daily': cfg.get('wrds', {}).get('daily_table_candidates', []),
}

for label, tables in candidate_groups.items():
    print(f'\n{label.upper()} table candidates in schema={schema!r}')
    for table in tables:
        exists = client.table_exists(schema, table)
        print(f'  {table}: {exists}')
        if exists:
            cols = client.table_cols(schema, table)
            print('    first columns:', cols[:25])


NAMES table candidates in schema='crsp'
  stksecurityinfohist: True
    first columns: ['permno', 'secinfostartdt', 'secinfoenddt', 'securitybegdt', 'securityenddt', 'securityhdrflg', 'hdrcusip', 'hdrcusip9', 'cusip', 'cusip9', 'primaryexch', 'conditionaltype', 'exchangetier', 'tradingstatusflg', 'securitynm', 'shareclass', 'usincflg', 'issuertype', 'securitytype', 'securitysubtype', 'sharetype', 'securityactiveflg', 'delactiontype', 'delstatustype', 'delreasontype']
  stocknames: True
    first columns: ['permno', 'namedt', 'nameenddt', 'shrcd', 'exchcd', 'siccd', 'ncusip', 'ticker', 'comnam', 'shrcls', 'permco', 'hexcd', 'cusip', 'st_date', 'end_date', 'namedum']
  dsenames: True
    first columns: ['permno', 'namedt', 'nameendt', 'shrcd', 'exchcd', 'siccd', 'ncusip', 'ticker', 'comnam', 'shrcls', 'tsymbol', 'naics', 'primexch', 'trdstat', 'secstat', 'permco', 'compno', 'issuno', 'hexcd', 'hsiccd', 'cusip']

DAILY table candidates in schema='crsp'
  stkdlysecuritydata: True
    firs

In [8]:
PILOT_TICKERS = cfg.get('wrds', {}).get('tickers') or ['AAPL', 'MSFT', 'NVDA']
PILOT_START = '2024-01-01'
PILOT_END = '2024-03-31'

source = WRDSEquitySource(client, cfg)
pilot_df = source.fetch_daily_ohlcv(
    PILOT_TICKERS,
    start_date=PILOT_START,
    end_date=PILOT_END,
)

print('Pilot rows:', len(pilot_df))
pilot_df.head()

Pilot rows: 183


,Date,Ticker,Open,High,Low,Close,Volume
0,2024-01-02,AAPL,187.15,188.44,183.885,185.64,81752737.0
1,2024-01-03,AAPL,184.22,185.88,183.43,184.25,58136569.0
2,2024-01-04,AAPL,182.15,183.0872,180.88,181.91,71280275.0
3,2024-01-05,AAPL,181.99,182.76,180.17,181.18,62064040.0
4,2024-01-08,AAPL,182.085,185.6,181.5,185.56,58748031.0


In [9]:
summary = pilot_df.groupby('Ticker').agg(
    rows=('Date', 'size'),
    min_date=('Date', 'min'),
    max_date=('Date', 'max'),
    min_close=('Close', 'min'),
    max_close=('Close', 'max'),
)
summary.sort_index()

,rows,min_date,max_date,min_close,max_close
Ticker,,,,,
AAPL,61,2024-01-02,2024-03-28,169.0,195.18
MSFT,61,2024-01-02,2024-03-28,367.75,429.37
NVDA,61,2024-01-02,2024-03-28,475.69,950.02


In [10]:
raw_dir = Path(cfg.get('wrds', {}).get('raw_cache_dir', './data/raw')).resolve()
print('Raw WRDS cache dir:', raw_dir)
for path in sorted(raw_dir.glob('wrds_*')):
    print(' -', path.name)

if client is not None:
    client.close()
    print('WRDS connection closed.')

Raw WRDS cache dir: D:\PyCharmProjects\KRNN\data\raw
 - wrds_equity_daily.parquet
 - wrds_equity_ohlcv.parquet
 - wrds_extract_manifest.json
 - wrds_symbol_map.parquet
WRDS connection closed.


## Interpretation

- If the direct `psycopg2` cell fails, the problem is outside the KRNN codebase.
- If direct `psycopg2` succeeds but `WRDSClient` fails, we should refactor the project to use direct SQLAlchemy/psycopg2 instead of the `wrds` package.
- If both succeed, run `python main_pipeline.py` from the repo root.